# M7 — QLoRA fine-tune of Qwen2.5-3B-Instruct

Run this on Colab (T4) or Kaggle (2xT4) -- NOT on the local RTX 5050, since bitsandbytes has no working Blackwell (sm_120) build. See `docs/PROJECT_PLAN.md`, "GGUF via llama.cpp, never bitsandbytes".

Hyperparameters per the project plan: r=16, alpha=32, 4-bit NF4 base.

**Before running:** upload `coaching_pairs.jsonl` (from `ml/finetuning/generate_coaching_pairs.py`, run locally) using the file picker in the left sidebar, or the upload cell below.

**If you already ran an earlier version of the install cell in this session:** it may have upgraded torch and left the runtime in a broken state (torch/torchvision version mismatch). Do Runtime > Restart session (a full restart, not just re-running cells) before rerunning the install cell below -- otherwise the fix pins torch to whatever version the earlier broken install left behind, not Colab's original working one.

**If the install cell's CUDA assertion still fails after a clean restart:** check Runtime > Change runtime type has a T4 GPU selected.

In [ ]:
# Unpinned on purpose: exact versions pinned to this repo's local dev
# environment (transformers==4.47.1 etc) broke on Colab's current image --
# bitsandbytes had no matching CUDA binary and pulled in a triton version
# missing triton.ops. Do NOT touch torch: Colab's preinstalled torch has a
# local build tag (e.g. +cu128) that isn't a real version on public PyPI,
# so pinning or reinstalling it by version string always 404s pip -- and
# when that happens mid-line, pip aborts the WHOLE line, silently skipping
# every package listed after it (this is what broke bitsandbytes install
# earlier: torch==... failed, so bitsandbytes right after it never ran).
# Leave torch alone entirely and use --no-deps on bitsandbytes so it can't
# upgrade torch as a side effect either.
!pip install -q -U transformers peft trl accelerate datasets
!pip install -q -U --force-reinstall --no-cache-dir --no-deps bitsandbytes

# Sanity check before spending time downloading the model: fail fast and
# loud if CUDA isn't actually available, rather than silently falling back
# to a CPU path that would make 4-bit loading fail later with a much more
# confusing error. (bitsandbytes has no public COMPILED_WITH_CUDA flag in
# current releases -- it just uses torch.cuda.is_available() internally,
# so that's the correct thing to check here too.)
import torch
import bitsandbytes as bnb
assert torch.cuda.is_available(), (
    "No CUDA device visible to torch -- check Runtime > Change runtime type "
    "is set to a GPU (T4), then Runtime > Restart session and rerun this cell."
)
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}, bitsandbytes {bnb.__version__}")

In [ ]:
# Upload coaching_pairs.jsonl if not already present in the Colab filesystem.
import os
if not os.path.exists("coaching_pairs.jsonl"):
    from google.colab import files
    uploaded = files.upload()
    assert "coaching_pairs.jsonl" in uploaded, "Upload coaching_pairs.jsonl (produced by generate_coaching_pairs.py)"

In [ ]:
import json

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "qwen2.5-3b-coaching-lora"

SYSTEM_PROMPT = (
    "You are a warm, practical speaking-confidence coach for people with speech "
    "differences. You are not a clinician; never use diagnostic or treatment language. "
    "Give short, encouraging, concrete advice."
)

In [ ]:
pairs = []
with open("coaching_pairs.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} coaching pairs")
print(pairs[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def to_chat_text(pair):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pair["instruction"]},
        {"role": "assistant", "content": pair["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

texts = [to_chat_text(p) for p in pairs]
dataset = Dataset.from_dict({"text": texts})
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# trl moved dataset_text_field/max_seq_length (now max_length) off
# SFTTrainer's constructor and into SFTConfig -- current trl no longer
# accepts them as direct SFTTrainer kwargs (that raised the TypeError seen
# when this used a plain TrainingArguments + kwargs). SFTConfig is a drop-in
# superset of TrainingArguments plus these SFT-specific fields.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
# Zip and download the adapter for M8 (merge -> GGUF -> local llama.cpp).
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)

from google.colab import files
files.download(f"{OUTPUT_DIR}.zip")